# 09 Figures

Figures from data, finished runs and trained models, using the plotting functions in `deepvoid.figures` and the `Run` class in `deepvoid.runs`. Every figure is shown here and saved as a PNG in `deepvoid_runs/figures/`.

1. **Data**: dark matter, the subhalo fields at each tracer spacing, and the T-web classes, at one slice.
2. **Training curves** of the finished runs.
3. **Predictions** of one trained model: input, truth, predicted classes, void score, misclassified voxels.
4. **Summary figure** of what DeepVoid does, for the top of the README.

Runtime: **L4 GPU** with **High-RAM** (sections 3 and 4 run the model over the whole box once).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')
!rm -rf /content/deepvoid
!git clone -q "https://{token}@github.com/skumagai3/deepvoid.git" /content/deepvoid
!git -C /content/deepvoid remote set-url origin https://github.com/skumagai3/deepvoid.git
!pip -q install --force-reinstall --no-deps /content/deepvoid
del token
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings

In [ ]:
from pathlib import Path

import numpy as np

from deepvoid import Catalog
from deepvoid import figures as F
from deepvoid.report import Report
from deepvoid.runs import Run, find_runs

ROOT = Path('/content/drive/MyDrive/deepvoid_runs')
RUN = 'subhalo_A_smooth2ch_L10'            # the trained model for sections 3 and 4
INPUT = 'tng300-3/subhalos/L10'            # the field it predicts from
SPACING = 10                               # its tracer spacing, h^-1 Mpc

catalog = Catalog()
run = Run(ROOT / RUN, catalog)
test = run.regions['test']
AXIS, INDEX = test.axis, test.start + test.size // 2    # the middle of the test slab
BOX = catalog.info(INPUT)['box_size']
out = Report(ROOT / 'figures')
print(run, '| slice', 'xyz'[AXIS], '=', INDEX, '| saving to', out.folder)

## 1. Data

In [ ]:
fields = {'dark matter': catalog.load('tng300-3/dm').data}
for L in (3, 5, 7, 10):
    fields[f'subhalos, spacing {L} h⁻¹ Mpc'] = catalog.load(f'tng300-3/subhalos/L{L}').data
mask = catalog.load('tng300-3/mask').data

fig = F.fields_figure(fields, mask, INDEX, AXIS, box_size=BOX, thickness=2,
                      title=f'TNG300-3: dark matter, subhalo fields and T-web classes '
                            f'(slice {"xyz"[AXIS]} = {INDEX}, 0.8 h⁻¹ Mpc thick; colour: log10(ρ/ρ̄ + 0.01))')
out.show(fig, '1_data')
del fields

## 2. Training curves

Validation scores by epoch for every finished subhalo run. Runs trained before void MCC was logged are missing from that panel.

In [ ]:
runs = [r for r in find_runs(ROOT, catalog) if r.name.startswith('subhalo_') and r.finished]
print([r.name for r in runs])
fig = F.training_curves(runs, keys=('val_void_mcc', 'val_mcc', 'val_loss'),
                        title='Subhalo runs: validation scores by epoch')
out.show(fig, '2_training_curves')

## 3. Predictions of one model

The whole box is predicted once (a few minutes). The void threshold is the one chosen on the validation slab when the run was scored, if there is one.

In [ ]:
scores = run.predict(INPUT)                         # class scores, (n, n, n, 4)
tuned = run.scores()[INPUT].get('tuned', {})
threshold = tuned.get('void_threshold')
label = 'smoothed tracer field' if run.config['data']['smoothing'] else 'tracer field'

fig = F.prediction_figure(run.input(INPUT), mask, scores, INDEX, AXIS, threshold=threshold,
                          box_size=BOX, input_label=f'input: {label}',
                          title=f'{RUN} on {INPUT}: test slab, slice {"xyz"[AXIS]} = {INDEX}')
out.show(fig, '3_prediction')

## 4. Summary figure

Dark matter and its T-web classes (top), and the sparse tracers with DeepVoid's prediction from them (bottom), on the same test-slab slice. The note gives the test-slab void scores of this run.

To put it at the top of the README, download `deepvoid_runs/figures/deepvoid_summary.png` and add it to `docs/figures/` in the repository.

In [ ]:
from deepvoid.evaluate import decide

pred = decide(scores, threshold)
s = run.scores()[INPUT]
s = s.get('tuned', s)
note = (f'TNG300-3, held-out test slab · tracer spacing {SPACING} h⁻¹ Mpc · '
        f'void F1 {s["void_f1"]:.2f}, void MCC {s.get("void_mcc", float("nan")):.2f}')
fig = F.summary_figure(catalog.load('tng300-3/dm').data, mask,
                       catalog.load(INPUT).data, pred, INDEX, AXIS, box_size=BOX,
                       spacing=SPACING, tracer_depth=25, note=note)
out.show(fig, 'deepvoid_summary')